> **제11회 기출변형 문제** — 교재 533~559쪽
> 문제 533~535쪽 / 정답·해설 536~559쪽
>
> 노트북과 같은 위치에 데이터 파일(csv)을 두고 위에서부터 차례대로 실행하세요.

## 문제

제공된 학습용 데이터(train.csv)는 온라인 게임 플레이어의 활동 데이터이다. 학습용 데이터를 활용하여 유저 등급(User_Level)을 예측하는 모형을 개발하고, 모형을 평가용 데이터(test.csv)에 적용하여 유저 등급을 예측하시오. 예측 결과는 아래의 [유의사항]과 [제출 형식]을 준수하여, CSV 파일(result.csv)을 생성하는 코드를 제출하시오.

**[유의사항]**
① 예측 결과는 F1 Score(macro) 평가지표에 따라 평가함
② 타겟 변수(User_Level)는 0, 1, 2 총 3개의 클래스로 구성됨 (0: Low, 1: Medium, 2: High)

**[제출 형식]**
① CSV 파일명: result.csv
② 예측 유저 등급 컬럼명: pred
③ 제출 컬럼 개수: pred 컬럼 1개
④ 평가용 데이터 개수와 예측 결과 데이터 개수 일치: 903개

**[제공 데이터]**
- train.csv: 학습용 데이터, 약 2,614개
- test.csv: 평가용 데이터, 약 903개 (User_Level 컬럼 미제공)

| 컬럼명 | 설명 |
|---|---|
| id | 플레이어 ID |
| playtime_hours | 누적 플레이 시간(시간) |
| purchase_amount | 누적 결제 금액(원) |
| login_days_last30 | 최근 30일 접속일수 |
| avg_session_min | 평균 세션 시간(분) |
| guild_participation | 길드 가입 여부 (0/1) |
| friend_count | 친구 수 |
| User_Level | (train에만 존재) 유저 등급 — 0: Low, 1: Medium, 2: High |

### 문제 풀이 방향

2유형은 정답 하나를 구하는 것이 아니라, 학습 → 예측 → 제출파일 생성까지 이어지는 전체 파이프라인을 코드로 완성하는 것이 목표입니다. 이 문제는 게임 활동 지표로 3개 클래스 중 하나를 맞히는 다중분류이므로, 순서는 다음과 같습니다.

1. 데이터를 불러와 구조와 결측치, 클래스 비율을 확인한다
2. 결측치를 처리한다
3. 학습 데이터 일부를 떼어내 검증용으로 사용하여, 실제 채점 전에 모델의 성능을 미리 가늠해본다
4. 모델을 학습시키고 검증 데이터로 F1-macro 점수를 확인한다
5. 확인이 끝나면 전체 학습 데이터로 다시 학습시킨 뒤, 진짜 평가 데이터(test.csv)를 예측한다
6. 요구된 형식(pred 컬럼 1개, 903행)으로 result.csv를 저장한다

이번 데이터도 모든 독립변수가 이미 숫자형이라서, 범주형 인코딩 단계 없이 결측치 처리 이후 바로 모델링으로 넘어갈 수 있습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 구조 확인**

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
train.head()

(2614, 8) (903, 7)


,id,playtime_hours,purchase_amount,login_days_last30,avg_session_min,guild_participation,friend_count,User_Level
0,1115,166.3,32567.0,1,43.4,0,118,2
1,3210,7.9,2860.0,23,17.4,0,4,0
2,1375,24.0,4569.0,6,59.5,1,41,0
3,2411,33.7,20780.0,15,30.4,1,96,0
4,853,9.1,4156.0,0,7.6,1,181,0


train은 8개 컬럼(User_Level 포함), test는 7개 컬럼(User_Level 없음)입니다. 첫 5행만 봐도 purchase_amount와 avg_session_min에 결측치가 있을 가능성이 있는데, 정확한 개수는 다음 셀에서 확인하겠습니다.

**셀 2) 결측치 및 타겟 분포 확인**

In [3]:
print(train.isna().sum())
print(test.isna().sum())
print(train['User_Level'].value_counts())

id                      0
playtime_hours          0
purchase_amount        63
login_days_last30       0
avg_session_min        88
guild_participation     0
friend_count            0
User_Level              0
dtype: int64
id                      0
playtime_hours          0
purchase_amount        17
login_days_last30       0
avg_session_min        27
guild_participation     0
friend_count            0
dtype: int64
User_Level
0    1310
1     841
2     463
Name: count, dtype: int64


purchase_amount와 avg_session_min 두 컬럼에 결측치가 있습니다. `value_counts()`로 확인한 User_Level 분포를 보면 Low(0)가 가장 많고 High(2)가 가장 적어, 세 클래스의 비율이 균등하지 않다는 것을 알 수 있습니다.

**셀 3) 결측치 처리**

In [4]:
for col in ['purchase_amount', 'avg_session_min']:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

print(train.isna().sum().sum(), test.isna().sum().sum())

0 0


두 컬럼 모두 train에서 계산한 중앙값으로 train과 test 양쪽에 동일하게 채웠습니다. purchase_amount처럼 소수의 결제 유저가 매우 큰 금액을 지불할 수 있는(분포가 한쪽으로 치우친) 컬럼에서는 평균보다 중앙값이 대푯값으로 더 안정적입니다.

> 💡 결제 금액처럼 0이 많고 일부만 매우 큰 값을 가지는 컬럼은 평균이 왜곡되기 쉬운 대표적인 경우입니다. 이런 컬럼의 결측치를 채울 때는 평균보다 중앙값을 우선 고려하는 것이 안전합니다.

**셀 4) 학습용/검증용 데이터 분리**

In [5]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['id', 'User_Level'])
y = train['User_Level']
X_test = test.drop(columns=['id'])

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_tr.shape, X_val.shape)

(2091, 6) (523, 6)


`id`는 플레이어를 구분하는 값일 뿐 유저 등급과 관련이 없으므로 학습에 사용할 컬럼(X)에서 제외했습니다. 셀 2에서 확인했듯 User_Level의 클래스 비율이 고르지 않으므로, `stratify=y`를 지정해 학습용과 검증용 양쪽에 원본과 동일한 클래스 비율이 유지되도록 나눴습니다.

**셀 5) 모델 학습 및 검증 성능 확인**

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)

val_pred = model.predict(X_val)
print("검증 F1-macro:", f1_score(y_val, val_pred, average='macro'))

검증 F1-macro: 0.6866929109829413


`f1_score(..., average='macro')`는 세 클래스(Low/Medium/High) 각각에 대해 F1 점수를 따로 구한 뒤 단순 평균 낸 값입니다. 데이터 개수가 상대적으로 적은 High(2)를 잘 맞히는지도 Low(0)만큼 동등하게 반영된다는 뜻입니다.

**셀 6) 전체 학습 데이터로 재학습 후 test 예측**

In [7]:
model.fit(X, y)
pred = model.predict(X_test)

print(pred[:10])

[0 0 1 0 0 1 0 0 1 0]


셀 5에서는 검증을 위해 train 데이터의 80%만 사용해서 학습했습니다. 성능을 확인했으니, 이제는 가지고 있는 train 데이터를 전부 사용해서 다시 학습시켰습니다.

**셀 7) 제출 파일 생성**

In [8]:
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)

print(result.shape)
print(result.head())

(903, 1)
   pred
0     0
1     0
2     1
3     0
4     0


행 개수가 요구된 903개와 정확히 일치하는지 확인했습니다.

### 전체 코드

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 결측치 처리 (train 기준으로 계산한 값을 test에도 동일 적용)
for col in ['purchase_amount', 'avg_session_min']:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

X = train.drop(columns=['id', 'User_Level'])
y = train['User_Level']
X_test = test.drop(columns=['id'])

# 검증용 분리 후 성능 확인
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)
val_pred = model.predict(X_val)
print("검증 F1-macro:", f1_score(y_val, val_pred, average='macro'))

# 전체 데이터로 재학습 후 test 예측
model.fit(X, y)
pred = model.predict(X_test)

# 제출 파일 생성
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)
print(result.shape)

검증 F1-macro: 0.6866929109829413
(903, 1)


### 정리

이 문제는 결측치가 있는 컬럼이 두 개로 늘어난 것을 제외하면, 지금까지 다뤄온 2유형 다중분류 문제와 같은 골격(결측치 처리 → 검증 분리 → 학습 → 예측 → 제출)을 그대로 따랐습니다. 결제 금액처럼 분포가 치우친 컬럼에서는 중앙값 대체가 더 안정적이라는 것도 함께 확인했습니다.

이번 모델의 검증 F1-macro는 약 0.69 수준이었습니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

이번 문제처럼 결측치가 있는 컬럼이 두 개 이상일 때는, 컬럼명을 하나씩 나열하는 대신 코드로 자동 탐지해서 반복 처리하는 방식이 더 안전합니다. 실제 시험에서 결측치 컬럼의 개수나 이름이 예상과 다르게 나오더라도 같은 코드로 대응할 수 있기 때문입니다.

**변형 예시: 결측치가 있는 컬럼을 하나하나 나열하지 않고 자동으로 찾아서 처리하려면**

In [10]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

missing_cols = train.columns[train.isna().sum() > 0].tolist()
print("결측치가 있는 컬럼:", missing_cols)

for col in missing_cols:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

print(train.isna().sum().sum())

결측치가 있는 컬럼: ['purchase_amount', 'avg_session_min']
0


`train.columns[train.isna().sum() > 0]`는 컬럼별 결측치 개수를 센 뒤, 0보다 큰(결측치가 있는) 컬럼의 이름만 자동으로 골라줍니다. 이렇게 찾은 컬럼 목록을 그대로 반복문에 사용하면, 셀 3에서 `['purchase_amount', 'avg_session_min']`이라고 직접 나열했던 것과 동일한 결과를 얻으면서도, 데이터가 바뀌어도 코드를 고칠 필요가 없어집니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 분포가 치우친 컬럼의 결측치 처리 | 중앙값(median) 대체 | 평균보다 극단값의 영향을 덜 받음 |
| 결측치가 있는 컬럼을 자동으로 탐지 | `train.columns[train.isna().sum() > 0]` | 컬럼명을 미리 알지 못해도 코드로 찾아낼 수 있음 |
| 클래스 불균형 상태로 데이터 분리 | `stratify=y` | 학습용/검증용에 클래스 비율을 동일하게 유지 |
| 다중분류 평가 | `f1_score(..., average='macro')` | 클래스별 F1을 동일 가중치로 평균 (소수 클래스도 동등하게 반영) |
| 제출 파일 저장 | `to_csv('result.csv', index=False)` | index=False 없으면 불필요한 인덱스 컬럼이 추가됨 |